# First attempt at combining text detection and OCR

In [1]:
from zettelsortierung import WritingDetection
from zettelsortierung.Zettelwerk import Zettelsammlung
from zettelsortierung.OCR import PaddleOCR
from dotenv import load_dotenv
import os
import cv2

load_dotenv()

True

In [6]:
import openvino as ov

core = ov.Core()
print(core.available_devices)

['CPU', 'GPU']


In [ ]:
#from huggingface_hub import hf_hub_download
#
## Download Latin models
#det_path = hf_hub_download("monkt/paddleocr-onnx", "detection/v5/det.onnx")
#rec_path = hf_hub_download("monkt/paddleocr-onnx", "languages/latin/rec.onnx")
#dict_path = hf_hub_download("monkt/paddleocr-onnx", "languages/latin/dict.txt")

detection/v5/det.onnx:   0%|          | 0.00/88.0M [00:00<?, ?B/s]

languages/latin/rec.onnx:   0%|          | 0.00/7.86M [00:00<?, ?B/s]

dict.txt: 0.00B [00:00, ?B/s]

## Working example of using OpenVINO

In [ ]:
# Instructions from https://docs.openvino.ai/2025/openvino-workflow/running-inference.html
import openvino as ov
import numpy as np
import cv2



# Step 1. Create OpenVINO Runtime Core
core = ov.Core()

# Step 2. Compile the Model
path = '/home/jan/.cache/huggingface/hub/models--monkt--paddleocr-onnx/snapshots/7b02d0a30a07ba2b92ad1ff5a8941ae2c633de65/languages/latin/rec.onnx'
compiled_model = core.compile_model(path, 'AUTO')
input_layer = compiled_model.input(0)
output_layer = compiled_model.output(0)

# Step 3. Create an Inference Request
infer_request = compiled_model.create_infer_request()

def openvino(memory):
    # Create tensor from external memory
    #root = os.getenv('ZETTELSAMMLUNG_ROOT')
    #img_path = Zettelsammlung.collect_zettel(root, 10)[1].recto_file_path
    #memory = np.float32(cv2.imread(img_path))
    memory = np.float32([memory]) / 255.0
    memory = np.transpose(memory, (0, 3, 1, 2))
    input_tensor = ov.Tensor(array=memory, shared_memory=False) # for shared_memory=True, input must be contiguous

    #result = compiled_model(input_tensor)[output_layer]
    #return result

    # Set input tensor for model with one input
    infer_request.set_input_tensor(input_tensor)

    # Step 5. Start Inference
    infer_request.start_async()
    infer_request.wait()

    # Step 6. Process the Inference Results
    output = infer_request.get_output_tensor()
    output_buffer = output.data#[output_layer]
    return output_buffer

In [20]:
def load_character_dict(dict_path):
    with open(dict_path, "r", encoding="utf-8") as f:
        chars = [line.strip("\n") for line in f]
    chars.insert(0, "")  # CTC blank at index 0
    chars.insert(503, ' ')
    return chars

dict_path = '/home/jan/.cache/huggingface/hub/models--monkt--paddleocr-onnx/snapshots/7b02d0a30a07ba2b92ad1ff5a8941ae2c633de65/languages/latin/dict.txt'
characters = load_character_dict(dict_path)

In [21]:
def ctc_decode(preds, characters):
    """
    preds: numpy array [1, T, C]
    characters: list of vocab characters (with blank at index 0)
    """
    preds = preds[0]  # remove batch dimension

    # Argmax across character dimension
    pred_indices = np.argmax(preds, axis=1)

    text = []
    prev_idx = -1

    for idx in pred_indices:
        if idx != 0 and idx != prev_idx:  # 0 = blank
            text.append(characters[idx])
        prev_idx = idx

    return "".join(text)

In [33]:
root = os.getenv('ZETTELSAMMLUNG_ROOT')
sammlung = Zettelsammlung.collect_zettel(root, 100)#[25:50]
#image = cv2.imread(sammlung[3].recto_file_path)
#regions = WritingDetection.detect_code_regions(image=image)

14it [00:00, 89.37it/s]


In [40]:
for zettel in sammlung:
    image = cv2.imread(zettel.recto_file_path)
    regions = WritingDetection.detect_code_regions(image=image)

    labels = []
    pad_regions = []
    y_im, x_im, c_im = image.shape
    for region in regions:
        pad = 10
        x, y, w, h = region

        x1_adj = max(0, x-pad)
        y1_adj = max(0, y-pad)
        x2_adj = min(x_im, x+w+pad)
        y2_adj = min(y_im, y+h+pad)
        pad_region = (x1_adj, y1_adj, x2_adj-x1_adj, y2_adj-y1_adj)
        pad_regions.append(pad_region)

        im_region = image[y1_adj:y2_adj, x1_adj:x2_adj]
        #cv2.imshow(winname="region", mat=im_region)
        #cv2.waitKey(delay=0)
        #cv2.destroyAllWindows()

        w_resize = int((x2_adj - x1_adj) * (48 / (y2_adj-y1_adj)))
        shrunk = cv2.resize(im_region, (w_resize, 48))
        #cv2.imshow(winname="shrunk", mat=shrunk)
        #cv2.waitKey(delay=0)
        #cv2.destroyAllWindows()

        preds = openvino(shrunk)
        label = ctc_decode(preds, characters)
        print(label)
        #label = ocr_model.read(image=im_region)
        labels.append(label)

    #WritingDetection.display_labeled_boxes(image=image, boxes=pad_regions, labels=labels)

KkWb 1988, S.74
deBn) dp,eo, )
KkWb 1988, S.74
s. M. Gertnd
(M.B'ekes Miochen
A.
Gasrnd
,Woörfen, Lirbfon.
SW Oldenb,
TEgn2unt
fahiges "een"für ein nichd lebens-
doyt ;
nicht bekannt:
Wig Li
Puson, din winl playo

sile dab duf 1a. bigishi 1599.
rs.1, mn.
sinje Mrife fre si mi% 1ro omfrebe
1 79 mn 1, estig Ia chrirg]
soss  feleguoab:
Kel,2.
Bracht k.
DoraWl
Archiv für wesifälische Volkskunde
(Pio)
Ann St
edm Stochum
Schaunen
Wibb.
Past.drieb
War waor dar füör en
lim halw 6
loule.
de
fedrebbel an de
Kiienink kuemmen
Landois, Essinh!
S.226
=nmpn ift zufoffuin
Goirifps, Mauffue, Görmen.
1. Srebleel
Anlen A.
41
2j E"] foüppfar Gefpr] [Vs]
ninss Mabes] [cts]
S
Jsl Js
Lhs Sm Möller 1976 Ss.5
TEGTE
Frageb. 1 8
[vrspeberey
Sroublonre
BBe VO
Dollage ril
Arn Wa
Warstein E.
M
Bür,Verne
Bün Vn
7
(worl vo Tinbel)
Bri Nf
K.A.Müller
Arn Wa
Warstein H.
Frageb. 1 8
BbrVo
Vollage
Mt rio sind
ver Jenyfese Vi ane
Mes B
Bracht K.
Ae I  Ri
Gregory § 107

2) Ginfn (kintan) [M. yb1.
Sifh[M]
BEK Vh
(WmWb; Ellin

## Now using the class implementation

In [3]:
from zettelsortierung.OCR import PaddleOCR

dict_path = os.getenv('PADDLE_OCR_ONNX_DICT_PATH')
model = PaddleOCR()
characters = model.load_character_dict(dict_path)

root = os.getenv('ZETTELSAMMLUNG_ROOT')
sammlung = Zettelsammlung.collect_zettel(root, 100)[45:50]

for zettel in sammlung:
    image = cv2.imread(zettel.recto_file_path)
    regions = WritingDetection.detect_code_regions(image=image)

    labels = []
    pad_regions = []
    y_im, x_im, c_im = image.shape
    for region in regions:
        pad = 10
        x, y, w, h = region

        x1_adj = max(0, x-pad)
        y1_adj = max(0, y-pad)
        x2_adj = min(x_im, x+w+pad)
        y2_adj = min(y_im, y+h+pad)
        pad_region = (x1_adj, y1_adj, x2_adj-x1_adj, y2_adj-y1_adj)
        pad_regions.append(pad_region)

        im_region = image[y1_adj:y2_adj, x1_adj:x2_adj]

        w_resize = int((x2_adj - x1_adj) * (48 / (y2_adj-y1_adj)))
        shrunk = cv2.resize(im_region, (w_resize, 48))

        preds = model.predict(shrunk)
        label = model.ctc_decode(preds, characters)
        print(label)
        labels.append(label)

    WritingDetection.display_labeled_boxes(image=image, boxes=pad_regions, labels=labels)

8it [00:00, 63.12it/s]


Frageb. 577
Meinkenbrocht/tn
Arn ML


Frageb. 5 12
(en lg
/ Can
Shr.
aep
Frageb. 5/6
Frageb. 5 16
Liffob IHrin
Mi Dh
Frageb. 5 16
Badrfose Hr
Min Rd
